# Analyse Exploratoire — Projet Data Centre d'Assistance
## DU Big Data & Data Science — Université de Montpellier 2025-2026
### Équipe : Randriamisaina Tsiory-Fanomezana · SHIRALI POUR Amir

Ce notebook réalise l'analyse exploratoire complète du dataset final (`dataset_complet.csv`).

**Plan :**
1. Chargement et aperçu global
2. Analyse univariée — variables numériques (`analyse_var`)
3. Analyse univariée — variables catégorielles
4. Analyse bivariée — durée vs variables explicatives
5. Analyse temporelle et saisonnalité
6. Synthèse et conclusions

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

# Détection de la racine du projet
def _racine():
    rep = Path().resolve()
    while True:
        if any((rep / m).exists() for m in ['.git', 'requirements.txt']):
            return rep
        if rep.parent == rep:
            return Path().resolve().parent
        rep = rep.parent

RACINE = _racine()
print(f'Racine du projet : {RACINE}')

plt.rcParams.update({'figure.dpi': 100, 'font.size': 11})
sns.set_style('whitegrid')
BLEU = '#4472C4'; ORANGE = '#ED7D31'; VERT = '#70AD47'; ROUGE = '#C00000'

## 1. Chargement et aperçu global

In [ ]:
df = pd.read_csv(RACINE / 'data' / 'dataset_complet.csv', encoding='utf-8')
df['date.ouverture'] = pd.to_datetime(df['date.ouverture'], errors='coerce')

print(f'Dimensions : {df.shape[0]:,} lignes × {df.shape[1]} colonnes')
print(f'Période : {df["date.ouverture"].dt.year.min()} – {df["date.ouverture"].dt.year.max()}')
print(f'Agents uniques : {df["Matricule.de.traitement"].nunique():,}')
df.head(3)

In [ ]:
# Taux de valeurs manquantes par colonne
nan_pct = (df.isna().mean() * 100).sort_values(ascending=False)
nan_pct = nan_pct[nan_pct > 0]
print('Colonnes avec valeurs manquantes :')
print(nan_pct.to_string())

## 2. Analyse univariée — variables numériques

Fonction `analyse_var` : stats descriptives + histogramme + boxplot + test de normalité.

In [ ]:
def analyse_var(df, var, titre=None):
    """Analyse univariée complète d'une variable numérique."""
    serie = df[var].dropna()
    titre = titre or var

    print(f'=== {titre} ({len(serie):,} valeurs non-NaN) ===')
    print(serie.describe().round(2).to_string())
    print(f'  Asymétrie (skewness) : {serie.skew():.3f}')
    print(f'  Kurtosis             : {serie.kurt():.3f}')

    # Test de normalité (Shapiro sur échantillon)
    sample = serie.sample(min(5000, len(serie)), random_state=42)
    stat, p = stats.shapiro(sample)
    print(f'  Shapiro-Wilk         : W={stat:.4f}, p={p:.4e} → {"NON normale" if p < 0.05 else "normale"}')

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(titre, fontsize=13, fontweight='bold')

    # Histogramme
    p99 = serie.quantile(0.99)
    axes[0].hist(serie[serie <= p99], bins=50, color=BLEU, edgecolor='white')
    axes[0].axvline(serie.median(), color=ROUGE, linestyle='--', label=f'Médiane={serie.median():.1f}')
    axes[0].axvline(serie.mean(), color=ORANGE, linestyle='--', label=f'Moyenne={serie.mean():.1f}')
    axes[0].set_title('Distribution (p99)')
    axes[0].legend(fontsize=9)

    # Boxplot
    axes[1].boxplot(serie, vert=True, patch_artist=True,
                    boxprops=dict(facecolor=BLEU, alpha=0.7))
    axes[1].set_title('Boxplot')
    axes[1].set_ylabel(var)

    # QQ-plot
    stats.probplot(sample, dist='norm', plot=axes[2])
    axes[2].set_title('QQ-plot (normalité)')

    plt.tight_layout()
    plt.show()
    print()